In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 360
matplotlib.rcParams['text.usetex'] = True
matplotlib.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
# plt.style.use('dark_background')
import seaborn as sns

from sbi.inference import NPE
from sbi.analysis import pairplot
from sbi.utils import BoxUniform
from sbi.neural_nets import posterior_nn

import torch
torch.manual_seed(42)

In [2]:
base = Path('/pscratch/sd/v/vtorresg/quijotes/PowerSpectrum/FoF/latin_hypercube')
params_file = Path('/global/homes/v/vtorresg/quijotes-sbi/latin_hypercube_params.txt')

In [3]:
env_files = {env: (f'group_003_pk_halo_{env}_real_los2_CIC_N512_dk2kf_kmax0p5_mono.npz')
             for env in ['all', 'void', 'sheet', 'filament', 'knot', 'random_void']}
env_files['random_void'] = 'group_003_pk_random_void_real_los2_CIC_N512_dk2kf_kmax0p5_mono.npz'

cube_ids = np.arange(800)

In [4]:
names = ['Omega_m', 'Omega_b', 'h', 'n_s', 'sigma8']
labels = [r'$\Omega_m$', r'$\Omega_b$', r'$h$', r'$n_s$', r'$\sigma_8$']

In [5]:
def load_env_data(base, params_file, env_files, envs, cube_ids, kmin=0.01, kmax=0.5,
                  transform='log10'):
    base = Path(base)
    envs = [envs] if isinstance(envs, str) else list(envs)
    cube_ids = np.asarray(list(cube_ids), dtype=int)

    params = np.loadtxt(params_file, ndmin=2)

    blocks = []
    k_by_env, slices = {}, {}
    start = 0

    for env in envs:
        filename = env_files[env]
        rows = []
        reference = None

        for cube_id in tqdm(cube_ids, desc=f'read {env}'):
            path = base / str(cube_id) / filename

            with np.load(path, allow_pickle=False) as data:
                k = np.asarray(data['k_center'], dtype=float)
                pk = np.asarray(data['Pk0'], dtype=float)
                low = np.asarray(data['k_bin_low'], dtype=float)
                high = np.asarray(data['k_bin_high'], dtype=float)
            rows.append(pk)

            arrays = (k, low, high)
            if reference is None:
                    reference = tuple(a.copy() for a in arrays)
        k = reference[0]
        mask = (k >= kmin) & (k <= kmax)
        block = np.stack(rows)[:, mask]

        if transform == 'log10':
            block = np.log10(block)

        blocks.append(block)
        k_by_env[env] = k[mask]
        slices[env] = slice(start, start + block.shape[1])
        start += block.shape[1]

    theta_np = params[cube_ids]
    x_np = np.concatenate(blocks, axis=1)

    x = torch.tensor(x_np, dtype=torch.float32)
    theta = torch.tensor(theta_np, dtype=torch.float32)

    return {'x': x, 'theta': theta, 'cube_ids': cube_ids,
            'envs': envs, 'k': k_by_env, 'feature_slices': slices, 'transform': transform}

In [6]:
def run_sbi(data, n_train=450, split_seed=42, train_seed=42, sample_seed=123,
            hidden_features=128, num_transforms=5, n_samples=5000, batch_size=128,
            learning_rate=5e-4, validation_fraction=0.1, patience=30, max_epochs=1000):

    x, theta = data['x'], data['theta']
    n_cubes = len(x)

    indices = np.random.default_rng(split_seed).permutation(n_cubes)
    train_rows = indices[:n_train]
    test_rows = indices[n_train:]

    x_train, x_test = x[train_rows], x[test_rows]
    theta_train, theta_test = theta[train_rows], theta[test_rows]

    low = torch.tensor([0.1, 0.03, 0.5, 0.8, 0.6])
    high = torch.tensor([0.5, 0.07, 0.9, 1.2, 1.0])

    torch.manual_seed(train_seed)

    prior = BoxUniform(low=low, high=high)
    builder = posterior_nn(model='nsf',
                           hidden_features=hidden_features,
                           num_transforms=num_transforms,
                           z_score_x='independent',
                           z_score_theta='independent')
    inference = NPE(prior=prior, density_estimator=builder, device='cpu')
    density_estimator = inference.append_simulations(theta_train, x_train).train(training_batch_size=batch_size,
                                                                                 learning_rate=learning_rate,
                                                                                 validation_fraction=validation_fraction,
                                                                                 stop_after_epochs=patience,
                                                                                 max_num_epochs=max_epochs,
                                                                                 show_train_summary=True)
    posterior = inference.build_posterior(density_estimator)

    torch.manual_seed(sample_seed)
    truth = theta_test.numpy()
    means = np.empty_like(truth, dtype=float)
    stds = np.empty_like(means)
    intervals = np.empty((4, len(test_rows), 5))

    for j in tqdm(range(len(test_rows)), desc='test'):
        samples = posterior.sample((n_samples,), x=x_test[j], show_progress_bars=False).detach().cpu().numpy()

        means[j] = samples.mean(axis=0)
        stds[j] = samples.std(axis=0, ddof=1)
        intervals[:, j, :] = np.quantile(samples, [0.025, 0.16, 0.84, 0.975], axis=0)

    low95, low68, high68, high95 = intervals
    errors = means - truth

    summary = pd.DataFrame({'param': names,
                            'bias': errors.mean(axis=0),
                            'RMSE': np.sqrt((errors**2).mean(axis=0)),
                            'mean_posterior_std': stds.mean(axis=0),
                            'mean_68_interval_width': (high68 - low68).mean(axis=0),
                            'coverage_68': ((truth >= low68) & (truth <= high68)).mean(axis=0),
                            'coverage_95': ((truth >= low95) & (truth <= high95)).mean(axis=0)})

    print(f'Envs: {data['envs']} | '
          f'Input: {tuple(x.shape)} | '
          f'Train: {len(train_rows)} | Test: {len(test_rows)}')
    print(summary.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

    return {'posterior': posterior, 'inference': inference, 'data': data, 'x_test': x_test,
            'theta_test': theta_test, 'train_cube_ids': data['cube_ids'][train_rows],
            'test_cube_ids': data['cube_ids'][test_rows], 'truth': truth, 'means': means,
            'stds': stds, 'low68': low68, 'high68': high68, 'low95': low95, 'high95': high95,
            'summary': summary}

In [7]:
def plot_diagnostics(result):
    truth, means = result['truth'], result['means']
    low68, high68 = result['low68'], result['high68']

    fig, axes = plt.subplots(2, 3, figsize=(13, 7))

    for j, ax in enumerate(axes.ravel()[:5]):
        ax.vlines(truth[:, j], low68[:, j], high68[:, j], color='royalblue', alpha=0.25)
        ax.scatter(truth[:, j], means[:, j], s=12, color='royalblue', alpha=0.7)
        lower = min(truth[:, j].min(), low68[:, j].min())
        upper = max(truth[:, j].max(), high68[:, j].max())
        ax.plot([lower, upper], [lower, upper], 'k--')
        ax.set(xlabel='True value', ylabel='Mean posterior', title=labels[j])
        ax.grid(alpha=0.2)

    axes.ravel()[-1].axis('off')
    fig.suptitle(' + '.join(result['data']['envs']))
    fig.tight_layout()
    plt.show()

    summary = result['summary']
    positions = np.arange(5)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(positions - 0.18, summary['coverage_68'], width=0.36, label='Intervalo 68 %', color='royalblue')
    ax.bar(positions + 0.18, summary['coverage_95'], width=0.36, label='Intervalo 95 %', color='darkorange')
    
    ax.axhline(0.68, color='royalblue', linestyle='--')
    ax.axhline(0.95, color='darkorange', linestyle='--')
    
    ax.set_xticks(positions, labels)
    ax.set(ylim=(0, 1), ylabel='Fraction containing the true value')
    ax.legend()
    fig.tight_layout()
    plt.show()

In [8]:
def plot_posterior(result, idx=0, n_samples=50_000, seed=123):
    torch.manual_seed(seed)
    samples = result['posterior'].sample((n_samples,), x=result['x_test'][idx], show_progress_bars=False).detach().cpu().numpy()
    truth = result['truth'][idx]
    means = samples.mean(axis=0)

    fig, axes = plt.subplots(2, 3, figsize=(13, 7))

    for j, ax in enumerate(axes.ravel()[:5]):
        ax.hist(samples[:, j], bins=60, density=True, color='royalblue', alpha=0.65)
        ax.axvline(truth[j], color='black', linestyle='--', label=f'True: {truth[j]:.4f}')
        ax.axvline(means[j], color='darkorange', label=f'Mean posterior: {means[j]:.4f}')
        ax.set(xlabel=labels[j], ylabel='PDF')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.2)

    axes.ravel()[-1].axis('off')
    cube_id = result['test_cube_ids'][idx]
    fig.suptitle(f'{' + '.join(result['data']['envs'])} — cube {cube_id}')
    fig.tight_layout()
    plt.show()

## Void

In [9]:
data_void = load_env_data(base, params_file, env_files, envs=['void'], cube_ids=cube_ids)
result_void = run_sbi(data_void, n_train=720, train_seed=42)

read void:   0%|          | 0/800 [00:00<?, ?it/s]

 Neural network successfully converged after 203 epochs.
        -------------------------
        ||||| ROUND 1 STATS |||||:
        -------------------------
        Epochs trained: 203
        Best validation performance: -11.8578
        -------------------------
        


train:   0%|          | 0/80 [00:00<?, ?it/s]

/pscratch/sd/v/vtorresg/venvs/sbi/lib/python3.12/site-packages/nflows/transforms/lu.py:80: UserWarning: torch.triangular_solve is deprecated in favor of torch.linalg.solve_triangularand will be removed in a future PyTorch release.
torch.linalg.solve_triangular has its arguments reversed and does not return a copy of one of the inputs.
X = torch.triangular_solve(B, A).solution
should be replaced with
X = torch.linalg.solve_triangular(A, B). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/BatchLinearAlgebra.cpp:2256.)
  outputs, _ = torch.triangular_solve(


Envs: ['void'] | Input: (800, 39) | Train: 720 | Test: 80
  param    bias   RMSE  mean_posterior_std  mean_68_interval_width  coverage_68  coverage_95
Omega_m -0.0094 0.0493              0.0465                  0.0971       0.7125       0.9500
Omega_b -0.0001 0.0106              0.0111                  0.0258       0.6875       0.9250
      h  0.0040 0.1210              0.1087                  0.2492       0.5250       0.8875
    n_s  0.0097 0.0892              0.0886                  0.1933       0.6375       0.9625
 sigma8 -0.0046 0.0293              0.0297                  0.0611       0.6500       0.9375


In [10]:
# plot_diagnostics(result_void)
# plot_posterior(result_void, idx=0)

## Sheet

In [11]:
data_sheet = load_env_data(base, params_file, env_files, envs=['sheet'], cube_ids=cube_ids)
result_sheet = run_sbi(data_sheet, n_train=720, train_seed=42)

read sheet:   0%|          | 0/800 [00:00<?, ?it/s]

 Neural network successfully converged after 227 epochs.
        -------------------------
        ||||| ROUND 1 STATS |||||:
        -------------------------
        Epochs trained: 227
        Best validation performance: -12.9989
        -------------------------
        


train:   0%|          | 0/80 [00:00<?, ?it/s]

Envs: ['sheet'] | Input: (800, 39) | Train: 720 | Test: 80
  param    bias   RMSE  mean_posterior_std  mean_68_interval_width  coverage_68  coverage_95
Omega_m -0.0116 0.0501              0.0441                  0.0923       0.7000       0.9500
Omega_b -0.0000 0.0108              0.0112                  0.0259       0.6875       0.9500
      h  0.0032 0.1193              0.1081                  0.2462       0.5625       0.8750
    n_s  0.0125 0.0995              0.0893                  0.1942       0.5375       0.9625
 sigma8 -0.0053 0.0313              0.0286                  0.0602       0.5750       0.9375


## Filament

In [12]:
data_fil = load_env_data(base, params_file, env_files, envs=['filament'], cube_ids=cube_ids)
result_fil = run_sbi(data_fil, n_train=720, train_seed=42)

read filament:   0%|          | 0/800 [00:00<?, ?it/s]

 Neural network successfully converged after 145 epochs.
        -------------------------
        ||||| ROUND 1 STATS |||||:
        -------------------------
        Epochs trained: 145
        Best validation performance: -11.8210
        -------------------------
        


train:   0%|          | 0/80 [00:00<?, ?it/s]

Envs: ['filament'] | Input: (800, 39) | Train: 720 | Test: 80
  param    bias   RMSE  mean_posterior_std  mean_68_interval_width  coverage_68  coverage_95
Omega_m -0.0074 0.0424              0.0386                  0.0788       0.6875       0.9500
Omega_b -0.0003 0.0104              0.0109                  0.0251       0.6875       0.9375
      h -0.0020 0.1191              0.1065                  0.2414       0.5375       0.8875
    n_s  0.0097 0.0996              0.0893                  0.1957       0.5625       0.9375
 sigma8 -0.0036 0.0408              0.0352                  0.0754       0.5500       0.9250


## Knot

In [13]:
data_knot = load_env_data(base, params_file, env_files, envs=['knot'], cube_ids=cube_ids)
result_knot = run_sbi(data_knot, n_train=720, train_seed=42)

read knot:   0%|          | 0/800 [00:00<?, ?it/s]

 Neural network successfully converged after 71 epochs.
        -------------------------
        ||||| ROUND 1 STATS |||||:
        -------------------------
        Epochs trained: 71
        Best validation performance: -9.5092
        -------------------------
        


train:   0%|          | 0/80 [00:00<?, ?it/s]

Envs: ['knot'] | Input: (800, 39) | Train: 720 | Test: 80
  param    bias   RMSE  mean_posterior_std  mean_68_interval_width  coverage_68  coverage_95
Omega_m -0.0076 0.0476              0.0437                  0.0889       0.6875       0.9500
Omega_b  0.0001 0.0105              0.0112                  0.0259       0.7125       0.9500
      h  0.0021 0.1215              0.1090                  0.2488       0.5375       0.8750
    n_s  0.0082 0.1100              0.1015                  0.2257       0.6375       0.9250
 sigma8 -0.0040 0.0491              0.0502                  0.0972       0.6625       0.9625


## Void + Sheet + Filament + Knot

In [14]:
data_env = load_env_data(base=base, params_file=params_file, env_files=env_files,
                         envs=['void', 'sheet', 'filament', 'knot'], cube_ids=cube_ids,
                         kmin=0.01, kmax=0.5)
result_env = run_sbi(data_env, n_train=720, train_seed=42)

read void:   0%|          | 0/800 [00:00<?, ?it/s]

read sheet:   0%|          | 0/800 [00:00<?, ?it/s]

read filament:   0%|          | 0/800 [00:00<?, ?it/s]

read knot:   0%|          | 0/800 [00:00<?, ?it/s]

 Neural network successfully converged after 270 epochs.
        -------------------------
        ||||| ROUND 1 STATS |||||:
        -------------------------
        Epochs trained: 270
        Best validation performance: -13.3063
        -------------------------
        


train:   0%|          | 0/80 [00:00<?, ?it/s]

Envs: ['void', 'sheet', 'filament', 'knot'] | Input: (800, 156) | Train: 720 | Test: 80
  param    bias   RMSE  mean_posterior_std  mean_68_interval_width  coverage_68  coverage_95
Omega_m -0.0058 0.0399              0.0376                  0.0770       0.6875       0.9375
Omega_b -0.0007 0.0101              0.0107                  0.0245       0.7125       0.9375
      h -0.0001 0.1115              0.1061                  0.2425       0.5875       0.8875
    n_s  0.0052 0.0757              0.0752                  0.1571       0.6500       0.9625
 sigma8 -0.0041 0.0289              0.0276                  0.0560       0.6375       0.9500


## All + Void + Sheet + Filament + Knot

In [15]:
data_env_all = load_env_data(base=base, params_file=params_file, env_files=env_files,
                         envs=['all', 'void', 'sheet', 'filament', 'knot'], cube_ids=cube_ids,
                         kmin=0.01, kmax=0.5)
result_env_all = run_sbi(data_env_all, n_train=720, train_seed=42)

read all:   0%|          | 0/800 [00:00<?, ?it/s]

read void:   0%|          | 0/800 [00:00<?, ?it/s]

read sheet:   0%|          | 0/800 [00:00<?, ?it/s]

read filament:   0%|          | 0/800 [00:00<?, ?it/s]

read knot:   0%|          | 0/800 [00:00<?, ?it/s]

 Neural network successfully converged after 232 epochs.
        -------------------------
        ||||| ROUND 1 STATS |||||:
        -------------------------
        Epochs trained: 232
        Best validation performance: -12.7898
        -------------------------
        


train:   0%|          | 0/80 [00:00<?, ?it/s]

Envs: ['all', 'void', 'sheet', 'filament', 'knot'] | Input: (800, 195) | Train: 720 | Test: 80
  param    bias   RMSE  mean_posterior_std  mean_68_interval_width  coverage_68  coverage_95
Omega_m -0.0072 0.0410              0.0386                  0.0797       0.6875       0.9625
Omega_b -0.0004 0.0097              0.0106                  0.0243       0.7250       0.9500
      h -0.0013 0.1122              0.1041                  0.2357       0.5125       0.8500
    n_s  0.0053 0.0810              0.0779                  0.1609       0.6125       0.9500
 sigma8 -0.0048 0.0314              0.0296                  0.0606       0.6750       0.9625


## Random voids

In [16]:
data_rand_void = load_env_data(base, params_file, env_files, envs=['random_void'], cube_ids=cube_ids)
result_rand_void = run_sbi(data_rand_void, n_train=720, train_seed=42)

read random_void:   0%|          | 0/800 [00:00<?, ?it/s]

 Neural network successfully converged after 132 epochs.
        -------------------------
        ||||| ROUND 1 STATS |||||:
        -------------------------
        Epochs trained: 132
        Best validation performance: -13.5228
        -------------------------
        


train:   0%|          | 0/80 [00:00<?, ?it/s]

Envs: ['random_void'] | Input: (800, 39) | Train: 720 | Test: 80
  param    bias   RMSE  mean_posterior_std  mean_68_interval_width  coverage_68  coverage_95
Omega_m -0.0100 0.0441              0.0393                  0.0812       0.7125       0.9375
Omega_b -0.0001 0.0104              0.0108                  0.0250       0.7125       0.9375
      h  0.0030 0.1150              0.1067                  0.2414       0.5625       0.8875
    n_s  0.0092 0.0909              0.0880                  0.1931       0.6125       0.9625
 sigma8 -0.0063 0.0290              0.0279                  0.0590       0.6125       0.9750


## Random voids + Void + Sheet + Filament + Knot

In [17]:
d = load_env_data(base, params_file, env_files, envs=['void', 'sheet', 'filament', 'knot', 'random_void'], cube_ids=cube_ids)
result_d = run_sbi(d, n_train=720, train_seed=42)

read void:   0%|          | 0/800 [00:00<?, ?it/s]

read sheet:   0%|          | 0/800 [00:00<?, ?it/s]

read filament:   0%|          | 0/800 [00:00<?, ?it/s]

read knot:   0%|          | 0/800 [00:00<?, ?it/s]

read random_void:   0%|          | 0/800 [00:00<?, ?it/s]

 Neural network successfully converged after 215 epochs.
        -------------------------
        ||||| ROUND 1 STATS |||||:
        -------------------------
        Epochs trained: 215
        Best validation performance: -13.8682
        -------------------------
        


train:   0%|          | 0/80 [00:00<?, ?it/s]

Envs: ['void', 'sheet', 'filament', 'knot', 'random_void'] | Input: (800, 195) | Train: 720 | Test: 80
  param    bias   RMSE  mean_posterior_std  mean_68_interval_width  coverage_68  coverage_95
Omega_m -0.0050 0.0318              0.0302                  0.0609       0.7250       0.9250
Omega_b  0.0001 0.0096              0.0105                  0.0240       0.6875       0.9750
      h -0.0002 0.1096              0.1016                  0.2280       0.5375       0.8750
    n_s  0.0096 0.0580              0.0524                  0.1017       0.5875       0.9375
 sigma8 -0.0046 0.0187              0.0172                  0.0321       0.6250       0.9375


## All + Random voids + Void + Sheet + Filament + Knot

In [18]:
d = load_env_data(base, params_file, env_files, envs=['all', 'void', 'sheet', 'filament', 'knot', 'random_void'], cube_ids=cube_ids)
result_d = run_sbi(d, n_train=720, train_seed=42)

read all:   0%|          | 0/800 [00:00<?, ?it/s]

read void:   0%|          | 0/800 [00:00<?, ?it/s]

read sheet:   0%|          | 0/800 [00:00<?, ?it/s]

read filament:   0%|          | 0/800 [00:00<?, ?it/s]

read knot:   0%|          | 0/800 [00:00<?, ?it/s]

read random_void:   0%|          | 0/800 [00:00<?, ?it/s]

 Neural network successfully converged after 262 epochs.
        -------------------------
        ||||| ROUND 1 STATS |||||:
        -------------------------
        Epochs trained: 262
        Best validation performance: -13.9406
        -------------------------
        


train:   0%|          | 0/80 [00:00<?, ?it/s]

Envs: ['all', 'void', 'sheet', 'filament', 'knot', 'random_void'] | Input: (800, 234) | Train: 720 | Test: 80
  param    bias   RMSE  mean_posterior_std  mean_68_interval_width  coverage_68  coverage_95
Omega_m -0.0067 0.0316              0.0294                  0.0588       0.7500       0.9125
Omega_b  0.0002 0.0096              0.0103                  0.0234       0.7125       0.9500
      h  0.0021 0.1042              0.1001                  0.2211       0.5625       0.8625
    n_s  0.0105 0.0546              0.0501                  0.0980       0.6125       0.9375
 sigma8 -0.0050 0.0186              0.0161                  0.0308       0.6000       0.9125


## All

In [19]:
d = load_env_data(base, params_file, env_files, envs=['all'], cube_ids=cube_ids)
result_d = run_sbi(d, n_train=720, train_seed=42)

read all:   0%|          | 0/800 [00:00<?, ?it/s]

 Neural network successfully converged after 191 epochs.
        -------------------------
        ||||| ROUND 1 STATS |||||:
        -------------------------
        Epochs trained: 191
        Best validation performance: -13.1815
        -------------------------
        


train:   0%|          | 0/80 [00:00<?, ?it/s]

Envs: ['all'] | Input: (800, 39) | Train: 720 | Test: 80
  param    bias   RMSE  mean_posterior_std  mean_68_interval_width  coverage_68  coverage_95
Omega_m -0.0113 0.0484              0.0440                  0.0919       0.6625       0.9500
Omega_b -0.0003 0.0108              0.0111                  0.0260       0.7125       0.9500
      h  0.0006 0.1210              0.1078                  0.2459       0.5250       0.8625
    n_s  0.0126 0.1019              0.0906                  0.2000       0.5500       0.9625
 sigma8 -0.0064 0.0385              0.0343                  0.0737       0.5625       0.9625
